# 01 · ICNALE GRA — build the pool

*Holistic essay score band (Low / Mid / High)*

### Where this sits

```
▶ 01 build the pool  →  02 sample  →  03 annotate  →  04 prompt  →  05 report
```

You run **01 once per group**, for your own track only. It ends by writing `data/pools/<track>_pool.json` — the file notebook 02 opens.

---

**What it is.** Asian-learner L2 English essays, each rated on holistic and analytic scales by many trained raters. This is an **automated writing evaluation** task: whole essays, not sentences.

**Difficulty of the labeling judgment:** ★★☆ — moderate, but a different shape of task: long texts and an ordered scale.

**Licence:** ⚠️ **Research use only — NOT redistributable.** Requires registration. Nothing derived from it may be committed to git or included in your submission bundle.  
**Cite:** Ishikawa, S. *The ICNALE Global Rating Archives.*

---

Every dataset in this course is reshaped into the **same canonical schema**, so one pipeline works for all of them:

```json
[{"id": 1, "text": "...", "label": "..."}]
```

The *raw* data, though, looks different every time. **That difference is the lesson** — half of building a gold standard is getting messy real data into a clean, consistent shape.

> The reshaping code below is read straight out of `scripts/reshape.py` — it is the same code `scripts/prep_datasets.py` runs, not a copy of it. What is *missing* from it is missing on purpose: the ✏️ cells are the decisions, and they are yours. (Generated by `scripts/_generate_pool_notebooks.py`; edit that or `reshape.py`, never the `.ipynb`.)

## Step 1 — Get the data (this one is manual)

ICNALE GRA is released for research use behind a registration form that emails you a password. There is nothing to automate, and that is deliberate — the licence does not permit redistribution.

1. Register at <https://language.sakura.ne.jp/icnale/download.html> and wait for the password.
2. Download and unpack `ICNALE_GRA_2.x.zip`.
3. From its rating tables, export a CSV with **exactly two columns**, `text` and `score`, and upload it here (or put it in `data/raw/icnale/essays_scores.csv`).

In Colab, the cell below opens a file picker.

In [ ]:
# In Colab: uncomment to upload your essays_scores.csv
# from google.colab import files; files.upload()

RAW_FILE = "essays_scores.csv"

## Step 2 — Look at the raw format

The cell prints the **distribution** of the scores, not just a couple of rows. You need that before step 3: it is what tells you where cutting the scale leaves you with three usable classes rather than one big one and two nearly empty ones.

In [ ]:
import csv

scores = []
with open(RAW_FILE, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    print("columns:", reader.fieldnames)
    for row in reader:
        try:
            scores.append(float(row["score"]))
        except (TypeError, ValueError):
            pass

scores.sort()
print(len(scores), "scores · min", scores[0], "· max", scores[-1])
for q in (10, 25, 33, 50, 67, 75, 90):
    print("   ", str(q) + "th percentile:", scores[int(len(scores) * q / 100)])

## Step 3 — Reshape into the canonical schema

One decision, and it is entirely yours: ✏️ **where do the band boundaries go?**

There is no right answer sitting in the data waiting to be found. Two honest ways to choose, and they disagree:

- **From the rubric** — if the scale you are using says what a Low essay is, use that. Your classes will come out uneven, possibly badly, but they mean something outside your own study.
- **From the distribution** — cut at the 33rd and 67th percentiles (printed above) and your classes come out balanced. Your F1 is then easier to read, and your bands mean nothing except "bottom third of this sample".

Pick one, say which in `PLAN.md`, and report the boundaries as numbers. A band definition that exists only as an unexplained `4.0` in a notebook is not a scheme.

⚠️ These labels are **ordered** (Low < Mid < High) but they are *not* alphabetical. Set `LABELS_ORDER = ["Low", "Mid", "High"]` in `config.py`, or the weighted κ gets computed over `High < Low < Mid`, which means nothing.

In [ ]:
import csv

def reid(items):
    """Renumber ids sequentially from 1, keeping the current order."""
    renumbered = []
    next_id = 1
    for item in items:
        new_item = dict(item)
        new_item["id"] = next_id
        renumbered.append(new_item)
        next_id = next_id + 1
    return renumbered

def reshape_icnale(csv_path, low_below=4.0, mid_below=7.0):
    """Band a numeric holistic score into Low / Mid / High.

    THE CUT-OFFS ARE PLACEHOLDERS. 4 and 7 are not from the ICNALE rubric - they are
    round numbers. Where you put the boundaries decides how hard the task is and how
    balanced the classes are, so set them from the rubric you are actually using and
    say what you chose in your report.
    """
    rows = []
    skipped = 0
    with open(csv_path, encoding="utf-8-sig", newline="") as handle:
        for record in csv.DictReader(handle):
            text = (record.get("text") or "").strip()
            raw_score = (record.get("score") or "").strip()
            if not text or not raw_score:
                continue
            try:
                score = float(raw_score)
            except ValueError:
                skipped = skipped + 1      # a non-numeric cell: report it, do not crash
                continue
            if score < low_below:
                label = "Low"
            elif score < mid_below:
                label = "Mid"
            else:
                label = "High"
            rows.append({"id": 0, "text": text, "label": label})
    if skipped:
        print("  note: skipped", skipped, "row(s) whose score cell was not a number.")
    return reid(rows)

In [ ]:
# ✏️ Step 3a · Cut the scale ─────────────────────────────────────
# Goal      : turn a numeric score into three bands, and be able to defend where.
# Shape     : rows = reshape_icnale(RAW_FILE, low_below=..., mid_below=...)
#             the defaults (4.0 / 7.0) are ROUND NUMBERS, not a rubric —
#             using them unchanged is a choice you would have to defend too
# Produce   : rows (a list)      ← later cells use this name
# Note      : run it a couple of ways and look at step 4 each time. Seeing
#             the counts move as you shift a boundary is the point.
# Careful   : whatever you settle on goes in PLAN.md as two numbers and a
#             reason. Do not re-cut after seeing your F1.

# ✏️ your code here


## Step 4 — Check the label balance

In [ ]:
from collections import Counter

print("total items:", len(rows))
print("label counts:", dict(Counter(item["label"] for item in rows)))
rows[:3]        # peek at the first three reshaped items

In [ ]:
# ✏️ Step 4b · React to the balance ──────────────────────────────
# Goal      : decide what the counts you just printed mean for your study.
# Shape     : MIN_PER_CLASS = <the size of your SMALLEST class>
#             that is the ceiling on N_PER_CLASS in config.py — a balanced
#             sample cannot draw more from a class than the class has
# Produce   : MIN_PER_CLASS (an int)      ← later cells use this name
# Note      : these counts are a direct function of the two numbers you just chose — if you do not like them, change the cuts NOW, not after step 5.
# Note      : if the rarest class is tiny, say so in PLAN.md. Merging it
#             away or living with fewer items are both defensible;
#             not noticing is not.

# ✏️ your code here


## Step 5 — Save it

⚠️ Keep this file **out of git** and **out of your submission bundle**. `.gitignore` and `scripts/make_submission.py` both exclude anything with `icnale` in the name — please leave that in place.

In [ ]:
# Save the pool. Two places you might want it:
#   * this repo, if you cloned it:  "../data/pools/icnale_pool.json"
#   * your Google Drive, so it survives the Colab runtime resetting
import json, pathlib

OUT_FILE = "../data/pools/icnale_pool.json"

# In Colab WITHOUT the repo, uncomment these two to write straight to Drive:
# from google.colab import drive; drive.mount("/content/drive")
# OUT_FILE = "/content/drive/MyDrive/icnale_pool.json"

pathlib.Path(OUT_FILE).parent.mkdir(parents=True, exist_ok=True)
with open(OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print("Saved", len(rows), "items to", OUT_FILE)

## What you just built, and what happens to it

This is the **pool** — everything usable in the corpus, with its natural label imbalance intact. It is **not** your gold set, and its labels are **not** your labels: they are the original corpus authors' judgment, and you have not yet agreed with them about anything.

What those labels are for is narrow, and worth being precise about:

1. **Stratifying the draw** in notebook 02 — you cannot sample evenly across classes without knowing what the classes are.
2. **A comparison** in notebook 03 — once you have annotated blind and adjudicated, `compare_to_published` shows you every item where your group landed somewhere different. That gap is evidence, and one of the more interesting things you can put in a report.

They are never the answer key you score the model against. That file does not exist yet — you make it in notebook 03.

---

**Next:** set `TRACK = "icnale"` in `config.py`, then open `02_sample.ipynb`.